# 02 - Push an EPC into a dataspace

Can I send a package (objects and arrays) to a server, see it listed, and read an array back? And does a session that did not announce writing refuse to write?

**This notebook writes.** It does nothing unless `RETP_WRITE_DATASPACE` names a dataspace you created for the test (and will delete). Running all the notebooks of this folder is therefore read-only by default.

## Before you run

Settings come from environment variables only. No server name, user or token is written in this notebook, and the token is never printed.

| Variable | Needed | Meaning |
|---|---|---|
| `RETP_URL` | yes | `wss://...` ETP endpoint |
| `RETP_TOKEN` | yes | bearer token, without its scheme |
| `RETP_INSECURE` | no | `1` skips the certificate check (default: checked) |
| `RETP_PARTITION` | no | OSDU data partition |
| `RETP_DATASPACE` | no | the dataspace to read; when unset, the first one listed is taken and the notebook says so |
| `RETP_WRITE_DATASPACE` | no | a dataspace you created for the test and will delete; only the writing notebook uses it, and it does nothing without it |
| `RETP_EPC` | no | path of an EPC to push (its `.h5` beside it) |

When a precondition is missing, a cell prints why and the rest of the notebook is skipped without an error. When the server answers something the ETP 1.2 specification forbids, an `assert` fails with a message. The assertions check the **shape** of the answers, never the values a given server holds.

In [ ]:
import re
import xml.etree.ElementTree as ET
import zipfile

import numpy as np
import retp
import _server

READ_KW = _server.connect_arguments()                     # for the read-only refusal below
KW = _server.connect_arguments(writing=True) if READ_KW else None
WRITE_DATASPACE = _server.write_dataspace() if KW else None
EPC = _server.epc_path() if WRITE_DATASPACE else None
READY = EPC is not None

In [ ]:
import re

UUID = re.compile(r"[0-9a-fA-F]{8}-(?:[0-9a-fA-F]{4}-){3}[0-9a-fA-F]{12}")
TYPE = re.compile(r"/([a-z]+[0-9]+)\.([A-Za-z0-9_]+)\(")


def uuid_of(uri):
    found = UUID.findall(uri)
    return found[-1].lower() if found else None


def family(uri):
    found = TYPE.search(uri)
    return found.group(1) if found else None


def qualified_type(uri):
    found = TYPE.search(uri)
    return f"{found.group(1)}.{found.group(2)}" if found else None


def refused(call, *expected):
    # Run call(); it must raise one of expected. Anything else is a failure.
    try:
        call()
    except expected as error:
        return error
    except Exception as other:
        raise AssertionError(f"expected {[e.__name__ for e in expected]}, got {type(other).__name__}: {other}")
    raise AssertionError("the call returned instead of refusing")

## A read-only session refuses to write, by name

A session is read-only unless it is opened with `writing=True` (the Transaction protocol is then never announced). `put_objects` and `push_epc` raise `retp.Unsupported` before anything is sent, so this cell writes nothing even though it names a dataspace.

In [ ]:
if READ_KW:
    with retp.connect(**READ_KW) as reader:
        assert reader.writing is False
        error = refused(lambda: reader.push_epc("no-such-file.epc", "refusal-check"), retp.Unsupported)
        assert "read-only" in str(error) and "writing=True" in str(error), f"the refusal does not name the rule: {error}"
        print("push_epc ->", type(error).__name__, "|", error)
        error = refused(lambda: reader.put_objects("refusal-check", []), retp.Unsupported)
        print("put_objects ->", type(error).__name__)

## What the package holds, read without retp

The expected identities come from the EPC itself, with the standard library only (`zipfile` and `xml`), so the check afterwards compares the server with something independent of `retp`. An object part is an XML part whose root carries a `uuid` attribute.

In [ ]:
def epc_uuids(path):
    found = {}
    with zipfile.ZipFile(path) as package:
        for name in package.namelist():
            if not name.lower().endswith(".xml") or name.startswith("[Content_Types]") or "_rels/" in name or name.startswith("docProps/"):
                continue
            root = ET.fromstring(package.read(name))
            identifier = root.attrib.get("uuid")
            if identifier:
                found[identifier.lower()] = (name, root.tag.split("}")[-1])
    return found


EXPECTED = {}
if READY:
    EXPECTED = epc_uuids(EPC)
    assert EXPECTED, f"{EPC.name} holds no object part with a uuid attribute"
    print(f"{EPC.name}: {len(EXPECTED)} object(s)")

## Push

`push_epc(path, dataspace)` is read by Rust. It returns a report: `islands` (each with `state`, `uris`, `arrays`, `bytes`), `sent`, `complete`, `left_out`, `dangling`, `without_transactions`, `stopped`. When the push is not complete it raises `retp.PushIncomplete`, with the report in `.report`.

TODO(verify): the `.h5` of the EPC is looked for beside it (the energyml-rust bridge says so for `Epc.open`); `push_epc` is assumed to do the same.

In [ ]:
REPORT = None
if READY:
    with retp.connect(**KW) as session:
        assert session.writing is True
        before = {uuid_of(row["uri"]) for row in session.resources(WRITE_DATASPACE)["resources"]}
        if before:
            print(f"Note: {WRITE_DATASPACE!r} already held {len(before)} object(s); the check below tolerates them.")
        try:
            REPORT = session.push_epc(str(EPC), WRITE_DATASPACE)
        except retp.PushIncomplete as stopped:
            print(stopped)
            print(stopped.report)
            raise AssertionError("the push did not go through in full (report above)")

    assert set(REPORT) == {"islands", "sent", "complete", "left_out", "dangling", "without_transactions", "stopped"}, sorted(REPORT)
    assert REPORT["complete"] is True and REPORT["stopped"] is None and REPORT["left_out"] == []
    assert REPORT["sent"] == len(REPORT["islands"]), "not every island was committed"
    assert all(island["state"] == "committed" for island in REPORT["islands"])
    print(REPORT["sent"], "island(s) committed;", sum(i["arrays"] for i in REPORT["islands"]), "array(s),", sum(i["bytes"] for i in REPORT["islands"]), "byte(s)")
    print("without_transactions:", REPORT["without_transactions"])
    for left in REPORT["dangling"]:
        print("leaves the package:", left)

## Everything pushed is there

The dataspace is listed again; every uuid read from the EPC (and every uuid of the report) must be in the listing.

In [ ]:
PUSHED = []
if REPORT:
    with retp.connect(**KW) as session:
        listed = session.resources(WRITE_DATASPACE)["resources"]
    by_uuid = {uuid_of(row["uri"]): row["uri"] for row in listed}
    missing = sorted(set(EXPECTED) - set(by_uuid))
    assert not missing, f"{len(missing)} object(s) of the EPC are not listed after the push, first {missing[0]}"
    reported = {uuid_of(uri) for island in REPORT["islands"] for uri in island["uris"]}
    assert reported <= set(by_uuid), "the report names objects that the server does not list"
    assert reported == set(EXPECTED), "the report and the EPC do not name the same objects"
    PUSHED = [by_uuid[u] for u in EXPECTED]
    print(f"all {len(EXPECTED)} object(s) of the EPC are listed")

## An array, read back

The first pushed object that names an array (`PathInHdfFile` or `PathInExternalFile`) gives the path; the values read back must agree with the metadata.

In [ ]:
PATH_KEYS = ("PathInHdfFile", "PathInExternalFile")


def array_paths(node):
    found = []
    if isinstance(node, dict):
        found += [node[k] for k in PATH_KEYS if isinstance(node.get(k), str) and node[k]]
        for value in node.values():
            found += array_paths(value)
    elif isinstance(node, list):
        for value in node:
            found += array_paths(value)
    return found


if PUSHED:
    with retp.connect(**KW) as session:
        objects, errors = session.get_objects(PUSHED)
        assert not errors, f"objects listed after the push cannot be read: {errors}"
        target = next(((uri, array_paths(o)[0]) for uri, o in objects.items() if array_paths(o)), None)
        if target is None:
            print("No pushed object names an array: nothing to read back.")
        else:
            uri, path = target
            found, errors = session.array_metadata(uri, [path])
            assert not errors, f"array_metadata failed after the push: {errors}"
            dims = list(found[0]["dimensions"])
            values, dimensions = session.read_array(uri, path)
            assert list(dimensions) == dims, f"{list(dimensions)} read, {dims} announced"
            assert values.size == int(np.prod(dims)), f"{values.size} values for dimensions {dims}"
            print("read back", path, dims, values.dtype)
elif READY:
    print("Nothing was pushed: no array to read back.")
print("done")

## Cleaning up

The notebook does not delete anything: the bridge has no call that creates or deletes a dataspace or an object (`crates/retp-python/DESIGN.md` lists them as not bound yet). Delete the dataspace with the server's own tools once you are done.